Imports

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

Token GitHub

In [2]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

print("Token carregado:", bool(GITHUB_TOKEN))

Token carregado: True


Funções de API

In [3]:
def fetch(url):
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"Erro {response.status_code}: {response.text}")
        return []

    try:
        return response.json()
    except:
        return []

Paginação

In [4]:
def fetch_all(url):
    results = []
    page = 1

    while True:
        paginated_url = f"{url}&page={page}"
        data = fetch(paginated_url)

        if not data:
            break

        results.extend(data)
        page += 1

    return results

Função principal

In [5]:
def build_github_df(owner, repo):

    contributors = fetch(f"https://api.github.com/repos/{owner}/{repo}/contributors")
    pulls = fetch_all(f"https://api.github.com/repos/{owner}/{repo}/pulls?state=all&per_page=100")
    issues = fetch_all(f"https://api.github.com/repos/{owner}/{repo}/issues?state=all&per_page=100")

    data = {}
    timeline = []

    # -----------------------------
    # COMMITS
    # -----------------------------
    for c in contributors:
        if "login" not in c:
            continue

        data[c["login"]] = {
            "commits": c.get("contributions", 0),
            "prs_opened": 0,
            "prs_merged": 0,
            "issues": 0,
            "last_activity": None
        }

    # -----------------------------
    # PRs
    # -----------------------------
    for pr in pulls:
        if not pr.get("user"):
            continue

        user = pr["user"]["login"]

        data.setdefault(user, {
            "commits": 0,
            "prs_opened": 0,
            "prs_merged": 0,
            "issues": 0,
            "last_activity": None
        })

        data[user]["prs_opened"] += 1

        if pr.get("merged_at"):
            data[user]["prs_merged"] += 1

        if pr.get("created_at"):
            date = pr["created_at"][:10]
            timeline.append({"date": date, "user": user})
            data[user]["last_activity"] = date

    # -----------------------------
    # ISSUES
    # -----------------------------
    for issue in issues:
        if "pull_request" in issue:
            continue

        if not issue.get("user"):
            continue

        user = issue["user"]["login"]

        data.setdefault(user, {
            "commits": 0,
            "prs_opened": 0,
            "prs_merged": 0,
            "issues": 0,
            "last_activity": None
        })

        data[user]["issues"] += 1

        if issue.get("created_at"):
            date = issue["created_at"][:10]
            timeline.append({"date": date, "user": user})
            data[user]["last_activity"] = date

    df = pd.DataFrame.from_dict(data, orient="index")

    # -----------------------------
    # FEATURE ENGINEERING
    # -----------------------------
    df["score"] = (
        df["commits"] +
        df["prs_opened"] * 2 +
        df["prs_merged"] * 5 +
        df["issues"]
    )

    df["merge_rate"] = df["prs_merged"] / df["prs_opened"].replace(0, 1)
    df["total_activity"] = df[["commits", "prs_opened", "issues"]].sum(axis=1)

    timeline_df = pd.DataFrame(timeline)

    return df, timeline_df

variaveis globais

In [6]:
owner = "ariannesuelen-lang"
repo = "Challenge_system"

Execução

In [7]:
df, timeline = build_github_df(owner, repo)

df.head()

,commits,prs_opened,prs_merged,issues,last_activity,score,merge_rate,total_activity
SarahPaoliello,13,1,1,4,2026-04-06,24,1.0,18
Andrew42400,3,0,0,0,None,3,0.0,3
taijarals,3,0,0,0,None,3,0.0,3
Jacksonfvsantos,2,0,0,0,None,2,0.0,2
Nosdan21,2,0,0,0,None,2,0.0,2


Top contribuidores

In [8]:
top10 = df.sort_values("score", ascending=False).head(10)
top10

,commits,prs_opened,prs_merged,issues,last_activity,score,merge_rate,total_activity
SarahPaoliello,13,1,1,4,2026-04-06,24,1.0,18
RyanSkg,0,2,2,0,2026-04-10,14,1.0,2
jgabriell-bispo,1,2,0,0,2026-03-30,5,0.0,3
Andrew42400,3,0,0,0,None,3,0.0,3
taijarals,3,0,0,0,None,3,0.0,3
ariannesuelen-lang,2,0,0,1,2026-04-04,3,0.0,3
Jacksonfvsantos,2,0,0,0,None,2,0.0,2
Nosdan21,2,0,0,0,None,2,0.0,2
CaiqueHighTech,1,0,0,0,None,1,0.0,1
Favel1nhah,1,0,0,0,None,1,0.0,1
